# RNN vs GRU vs LSTM on real text (IMDb sentiment)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudanum/MIA5150/blob/main/notebooks/sequential-models-rnn-gru-lstm/04_sentiment_imdb.ipynb)

In notebooks 01–03 you built recurrent networks by hand and watched gates fight the vanishing gradient. Now we put all three cores — vanilla **RNN**, **GRU**, and **LSTM** — to work on a real NLP task: **binary sentiment classification of IMDb movie reviews**. One pipeline, one trainer, three swappable recurrent cores, and a fair fight on parameters, accuracy, and speed.

**Learning goals**

- Build a complete classical NLP pipeline: **tokenize → vocab → embed → recurrent core → classify**.
- Implement **masked mean pooling** and understand why it beats "just take the last hidden state" on padded batches.
- Train and compare `nn.RNN`, `nn.GRU`, and `nn.LSTM` with identical data, budget, and recipe.
- Read a params / accuracy / speed trade-off table and connect it to what you saw in notebooks 02–03.
- Probe the trained models qualitatively — including a long review whose sentiment flips at the end.

## 1. The task: thumbs-up or thumbs-down?

The [IMDb dataset](https://huggingface.co/datasets/stanfordnlp/imdb) is 50,000 movie reviews, each labeled **positive** or **negative**. It is the "hello, world" of real-text classification: messy, informal, long documents — nothing like the toy sequences of notebooks 01–03.

### The pipeline

Every classical NLP classifier has the same skeleton:

$$\text{review} \xrightarrow{\text{tokenize}} \text{tokens} \xrightarrow{\text{vocab}} \text{ids} \xrightarrow{\text{embed}} \mathbf{e}_1,\dots,\mathbf{e}_T \xrightarrow{\text{recurrent core}} \mathbf{h}_1,\dots,\mathbf{h}_T \xrightarrow{\text{pool}} \bar{\mathbf{h}} \xrightarrow{\text{linear}} \text{logit} \xrightarrow{\sigma} p(\text{positive})$$

1. **Tokenize** — split raw text into word pieces.
2. **Vocabulary** — map each token to an integer id.
3. **Embed** — each id becomes a dense vector $\mathbf{e}_t \in \mathbb{R}^{64}$, *learned from scratch* on this task.
4. **Recurrent core** — an RNN, GRU, or LSTM reads the vectors left to right and produces a hidden state $\mathbf{h}_t \in \mathbb{R}^{128}$ per position. This is the only part we swap between runs.
5. **Pool + classify** — collapse the hidden states into one vector (masked mean), then a linear layer produces a single logit, trained with binary cross-entropy.

### Why this stack matters

Embeddings + recurrence + pooling was *the* NLP stack of roughly 2014–2018: sentiment, translation, and named-entity recognition all ran on some variant of it. Transformers keep the same skeleton and swap step 4 for attention, so understanding this pipeline end-to-end makes the modern stack legible — same shapes, same loss, different mixing mechanism.

In [ ]:
# The only package Colab does not preinstall for this notebook:
!pip -q install datasets

In [ ]:
import random
import re
import time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ---- Config: every tunable knob lives here ----
TRAIN_SUBSET = 5000   # balanced training reviews (half positive, half negative)
TEST_SUBSET = 2000    # balanced held-out reviews (our validation monitor)
VOCAB_SIZE = 20000    # keep this many of the most common TRAIN tokens
MAX_LEN = 256         # truncate reviews to this many tokens
EMB_DIM = 64          # embedding size
HID_DIM = 128         # recurrent hidden size
BATCH_SIZE = 64
EPOCHS = 3
LR = 1e-3             # Adam learning rate
CORES = ['rnn', 'gru', 'lstm']

## 2. Data: balanced subsets, a tiny tokenizer, and a vocabulary

**Balanced subsets, deterministically.** The full IMDb train/test splits are already balanced, but 25,000 reviews is more than we need for a CPU lab. We shuffle with a fixed seed and keep `TRAIN_SUBSET` training and `TEST_SUBSET` held-out reviews, **half positive and half negative** — no class imbalance to confound the comparison, and fully reproducible run to run.

**A deliberately simple tokenizer.** Lowercase the text, then the regex `[a-z']+` keeps words and apostrophes and drops numbers and punctuation. Production systems use subword tokenizers (you will meet BPE in the transformers module), but a word-level tokenizer keeps every stage of this pipeline inspectable.

**Vocabulary from TRAIN only.** We keep the 20,000 most common *training* tokens — building the vocab from test data would leak information about the test distribution. Two ids are reserved: `<pad>` = 0 (filler for short sequences) and `<unk>` = 1 (any token not in the vocab); real tokens start at id 2.

**Truncate at `MAX_LEN = 256`.** Longer reviews are rare and expensive: a recurrent core's cost grows linearly with sequence length, and 256 tokens covers most reviews.

In [ ]:
imdb = load_dataset('stanfordnlp/imdb')   # the classic IMDb review dataset (Maas et al., 2011)
print(imdb)

def balanced_subset(hf_split, total, seed=SEED):
    '''Deterministically pick `total` reviews, half positive and half negative.'''
    labels = hf_split['label']
    pos = [i for i, y in enumerate(labels) if y == 1]
    neg = [i for i, y in enumerate(labels) if y == 0]
    rng = random.Random(seed)
    rng.shuffle(pos)
    rng.shuffle(neg)
    keep = pos[: total // 2] + neg[: total // 2]
    rng.shuffle(keep)                      # interleave the classes
    return hf_split.select(keep)

train_raw = balanced_subset(imdb['train'], TRAIN_SUBSET)
test_raw = balanced_subset(imdb['test'], TEST_SUBSET, seed=SEED + 1)
print(f'train: {len(train_raw)} reviews ({sum(train_raw["label"])} positive), '
      f'test: {len(test_raw)} reviews ({sum(test_raw["label"])} positive)')

TOKEN_RE = re.compile(r"[a-z']+")

def tokenize(text):
    '''Lowercase word tokenizer: keeps letters and apostrophes, drops the rest.'''
    return TOKEN_RE.findall(text.lower())

# --- vocabulary from TRAIN tokens only ---
counter = Counter()
for text in train_raw['text']:
    counter.update(tokenize(text))

most_common = counter.most_common(VOCAB_SIZE)
itos = ['<pad>', '<unk>'] + [tok for tok, _ in most_common]
stoi = {tok: i for i, tok in enumerate(itos)}
PAD_ID, UNK_ID = 0, 1
print(f'vocab size: {len(itos)} (including <pad> and <unk>)')
print(f'10 most common tokens: {[tok for tok, _ in most_common[:10]]}')

def encode(text):
    '''Token -> id, unknown -> <unk>, truncate to MAX_LEN. Never returns empty.'''
    ids = [stoi.get(tok, UNK_ID) for tok in tokenize(text)][:MAX_LEN]
    return ids if ids else [UNK_ID]

# quick look at review lengths after encoding
sample_lengths = [len(encode(t)) for t in train_raw['text'][:500]]
print(f'encoded lengths (first 500 train reviews): '
      f'mean {np.mean(sample_lengths):.0f} tokens, max {max(sample_lengths)}')

### Padding and the collate function

`nn.RNN` and friends want rectangular tensors of shape `(batch, time, features)`, but reviews have different lengths. The fix is **padding**: fill shorter sequences with `<pad>` (id 0) up to the longest sequence *in the batch* — not a global maximum, since there is no point padding a batch of 30-word reviews out to 256. The collate function also returns each row's **true length**, which the model's pooling step uses to ignore the pad positions.

In [ ]:
class IMDbDataset(Dataset):
    '''Pre-encodes every review once; __getitem__ returns (token ids, label).'''

    def __init__(self, hf_split):
        self.examples = [(encode(t), y) for t, y in zip(hf_split['text'], hf_split['label'])]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

def collate_fn(batch):
    '''Pad to the longest sequence *in this batch*; also return true lengths.'''
    ids_list, labels = zip(*batch)
    lengths = torch.tensor([len(ids) for ids in ids_list], dtype=torch.long)
    padded = torch.zeros(len(ids_list), lengths.max().item(), dtype=torch.long)  # PAD_ID = 0
    for i, ids in enumerate(ids_list):
        padded[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.float)
    return padded, lengths, labels

train_loader = DataLoader(IMDbDataset(train_raw), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(IMDbDataset(test_raw), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_fn)

# sanity check: one batch
x, lengths, y = next(iter(train_loader))
print(f'batch x: {tuple(x.shape)}   lengths: {tuple(lengths.shape)}   y: {tuple(y.shape)}')
print(f'lengths in this batch: min {lengths.min().item()}, max {lengths.max().item()}')

## 3. The model: one net, three swappable cores

`SentimentNet(core)` wraps the whole pipeline after the vocabulary:

1. `nn.Embedding(|V|, 64)` — a lookup table $\mathbf{E} \in \mathbb{R}^{|V| \times 64}$, initialized randomly and learned on this task. `padding_idx=0` pins the `<pad>` vector at zero.
2. `nn.RNN` / `nn.GRU` / `nn.LSTM` with 64-dim inputs and 128 hidden units — the only component that changes between runs.
3. **Masked mean pooling** — average the hidden states over the *real* positions only, using the lengths the collate function returned:

$$\bar{\mathbf{h}} = \frac{1}{L}\sum_{t=1}^{L} \mathbf{h}_t \qquad \text{(pad positions excluded by the mask)}$$

4. `nn.Linear(128, 1)` → one logit. `BCEWithLogitsLoss` fuses sigmoid + binary cross-entropy in a numerically stable form.

### Why masked mean instead of "take the last hidden state"?

Two practical reasons:

- **Padding makes "the last state" ill-defined.** In a padded batch, row $i$'s real content ends at position $L_i$, and every row's $L_i$ is different. Grabbing $\mathbf{h}_T$ reads pad positions for short reviews; gathering each row's $\mathbf{h}_{L_i}$ works but is fiddly.
- **Even unpacked, the last state is a bottleneck.** Everything the network knows about the review must survive to the final step — and notebook 02 showed how hard that is for a vanilla RNN over long distances. The mean lets *every* position contribute evidence and gives gradients a direct, length-independent path to each position.

Sentiment is largely a *bag-of-evidence* task — a scattering of strong words (*masterful*, *tedious*, *sublime*) — so mean pooling is a good match. The exercises ask you to try last-hidden pooling and see which core suffers most.

### Where the parameters live

The **embedding matrix dominates**: about $20{,}000 \times 64 \approx 1.28$M parameters, identical for all three models. The cores differ only in the recurrent block — a vanilla RNN has one weight pair per step, a GRU has three (reset, update, candidate), an LSTM has four (input, forget, output, candidate) — so the **GRU's recurrent core has ~25% fewer parameters than the LSTM's**, even though the two totals look close.

In [ ]:
CORE_CLASSES = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}

class SentimentNet(nn.Module):
    '''Embedding -> recurrent core -> masked mean pooling -> linear head.'''

    def __init__(self, core, vocab_size, emb_dim=EMB_DIM, hid_dim=HID_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.rnn = CORE_CLASSES[core](emb_dim, hid_dim, batch_first=True)
        self.head = nn.Linear(hid_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)                                # (B, T, E)
        h, _ = self.rnn(emb)                                   # (B, T, H)
        steps = torch.arange(x.size(1), device=x.device)       # (T,)
        mask = (steps < lengths.unsqueeze(1)).unsqueeze(-1)    # (B, T, 1): True on real tokens
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)  # (B, H)
        return self.head(pooled).squeeze(-1)                   # (B,)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for core in CORES:
    m = SentimentNet(core, len(itos))
    print(f'{core.upper():4s}: {count_params(m):,} parameters')

## 4. Training: a fair fight

All three cores get the **same recipe** — the same seed (so the same weight-init regime and the same data order), Adam at $10^{-3}$, 3 epochs, batch size 64. Per epoch we report train accuracy and accuracy on the held-out balanced subset, which plays the role of a **validation monitor** here. (In real work you would keep a third, untouched test split; nothing in this lab is tuned on the monitor beyond eyeballing three curves.)

**What to expect, qualitatively.** GRU and LSTM usually land close together — the gates of notebook 03 doing their job — with the vanilla RNN a few points behind, and the GRU getting there with the leanest gated core. Exact numbers vary with seed and subset; the *ordering and the gap* are the signal, not the decimals.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            logits = model(x, lengths)
            correct += ((logits > 0).float() == y).sum().item()
            total += y.size(0)
    return correct / total

def train_model(core, epochs=EPOCHS):
    '''Train a SentimentNet from scratch; return (model, history, wall_seconds).'''
    torch.manual_seed(SEED)   # same init + same shuffle order for every core
    model = SentimentNet(core, len(itos)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    history = {'train_acc': [], 'val_acc': [], 'epoch_time': []}
    t0 = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        te0 = time.perf_counter()
        correct = total = 0
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x, lengths)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
            correct += ((logits.detach() > 0).float() == y).sum().item()
            total += y.size(0)
        train_acc = correct / total
        val_acc = accuracy(model, val_loader)
        epoch_time = time.perf_counter() - te0
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(epoch_time)
        print(f'{core.upper():4s} epoch {epoch}/{epochs} — '
              f'train acc {train_acc:.3f} — val acc {val_acc:.3f} — {epoch_time:.1f}s')
    wall = time.perf_counter() - t0
    print(f'{core.upper():4s} finished in {wall:.1f}s')
    return model, history, wall

In [ ]:
results = {}
for core in CORES:
    model, history, wall = train_model(core)
    results[core] = {'model': model, 'history': history,
                     'wall': wall, 'params': count_params(model)}

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for core in CORES:
    h = results[core]['history']
    epochs = range(1, len(h['val_acc']) + 1)
    ax.plot(epochs, h['val_acc'], marker='o', label=f'{core.upper()} (val)')
    ax.plot(epochs, h['train_acc'], marker='.', linestyle='--', alpha=0.5,
            color=ax.lines[-1].get_color(), label=f'{core.upper()} (train)')
ax.set_xlabel('epoch')
ax.set_ylabel('accuracy')
ax.set_title('IMDb sentiment: RNN vs GRU vs LSTM')
ax.set_xticks(range(1, EPOCHS + 1))
ax.set_ylim(0.5, 1.0)
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

> **What to notice.**
>
> - Train accuracy (dashed) should climb above val accuracy (solid) for all three cores — a mild, healthy dose of overfitting on 5,000 reviews.
> - GRU and LSTM should track each other closely; the vanilla RNN typically trails. If your RNN curve is close, remember it is also the cheapest core per epoch.
> - Three epochs is a snapshot, not a converged contest — the exercises invite you to run longer and change the recipe.

In [ ]:
print(f"{'core':6s} {'params':>12s} {'final val acc':>14s} {'s/epoch':>9s}")
print('-' * 44)
for core in CORES:
    r = results[core]
    s_per_epoch = r['wall'] / EPOCHS
    print(f"{core.upper():6s} {r['params']:>12,} "
          f"{r['history']['val_acc'][-1]:>14.3f} {s_per_epoch:>8.1f}s")

# the totals hide the interesting comparison, because the embedding dominates:
gru_core = sum(p.numel() for n, p in results['gru']['model'].named_parameters() if 'rnn' in n)
lstm_core = sum(p.numel() for n, p in results['lstm']['model'].named_parameters() if 'rnn' in n)
print(f'\nrecurrent-core params: GRU {gru_core:,} vs LSTM {lstm_core:,} — '
      f'the GRU core is {100 * (1 - gru_core / lstm_core):.0f}% leaner (3 gates vs 4)')

## 5. Inference: does it *read*?

Numbers aside, let's hand the models a few reviews we wrote ourselves. The first two are unambiguous (one glowing, one scathing); the third mixes its signals (*"Slow and quiet, but … beautiful"*); the fourth is a long one whose sentiment **flips at the end** — a glowing preamble runs for dozens of tokens before one word, *but*, reverses everything. Watch whether each core's probability reflects the reversal.

Keep in mind these are 3-epoch models trained on 5,000 reviews: treat the probabilities as rough readings, not calibrated judgments.

In [ ]:
def predict(text, core='lstm'):
    '''Return (P(positive), label) for a raw review string.'''
    model = results[core]['model']
    model.eval()
    enc = encode(text)
    ids = torch.tensor([enc], dtype=torch.long, device=device)
    lengths = torch.tensor([len(enc)], dtype=torch.long, device=device)
    with torch.no_grad():
        logit = model(ids, lengths)
    p = torch.sigmoid(logit).item()
    return p, ('POSITIVE' if p > 0.5 else 'NEGATIVE')

reviews = [
    "One of the best movies I have ever seen. The acting is superb, the story is gripping, "
    "and I loved every minute of it.",
    "A complete waste of time. The plot is a mess, the acting is awful, and I nearly fell "
    "asleep before the end.",
    "Slow and quiet, but the performances are beautiful and it stayed with me for days.",
    ("This film has so much going for it. The cinematography is gorgeous, the music is "
     "beautiful, the lead performance is the best of her career, the supporting cast is "
     "flawless, the first hour builds a world you never want to leave, and the set pieces "
     "are genuinely thrilling. I was ready to call it a masterpiece, I really was — "
     "but the final act ruined it for me."),
]

for core in CORES:
    print(f'--- {core.upper()} ---')
    for text in reviews:
        p, label = predict(text, core)
        print(f'  [{label:8s} p={p:.2f}] {text[:72]}{"…" if len(text) > 72 else ""}')
    print()

### Why is the flip hard?

To score that last review correctly, a model must (a) register the contrast marker *but* and (b) let the short negative ending outweigh dozens of tokens of positive setup. Two forces work against it:

- **Pooling dilutes.** Masked mean gives every position a vote, and the preamble casts many. The ending's evidence must be encoded *strongly* in the hidden states to swing the average.
- **Distance decays.** For the vanilla RNN, notebook 02's vanishing gradient means the training signal for long-range contrast is weak; the GRU and LSTM gates give the *but → negative ending* relation a more protected channel.

One honest caveat: a single hand-written review is a probe, not a benchmark. Depending on seed and subset, any of the three cores can get this one right or wrong — the point is the *failure mode*, not this particular score. Try shortening the preamble, or moving *but* earlier, and watch the probability move.

## 6. Where this goes next

Two pieces of this pipeline are deliberately old-fashioned:

- **Embeddings learned from scratch** on 5,000 reviews know little about language. Modern NLP starts from embeddings — or entire models — pre-trained on billions of words.
- **The recurrent core is the bottleneck**: strictly sequential (no parallelizing over time), and its memory of position 1 has been overwritten 255 times by the final step.

**Attention** replaces recurrence with direct pairwise interactions between positions: no long chain, fully parallel, and — combined with pre-training — the foundation of every modern LLM. The skeleton you just built survives intact: tokenize → embed → *mix* → pool → classify. Only the mixer changes. You will build that mixer in the transformers module.

## Try it yourself

1. **Bidirectional.** Add `bidirectional=True` to the recurrent core, and double the head's input (`nn.Linear(2 * HID_DIM, 1)` — the core now emits a forward and a backward state per position). What changes in the parameter count and the val accuracy? Why might reading a review *backwards* help sentiment?
2. **Deeper.** Pass `num_layers=2` to the LSTM (note: PyTorch only applies its `dropout` argument *between* stacked layers). Does the extra capacity help on 5,000 reviews, or does train accuracy run away while val accuracy stalls?
3. **Longer memory.** Set `MAX_LEN = 512`. How much does val accuracy move — and what happens to seconds/epoch? Sequence length is the recurrent core's price tag.
4. **Last-hidden pooling.** Replace masked mean with each row's true final state: `h[torch.arange(x.size(0)), lengths - 1]`. Which core suffers most, and how does that connect to the vanishing-gradient results of notebook 02?

## Recap

- You built the full classical NLP stack — **tokenize → vocab → embed → recurrent core → masked-mean pool → logit** — and trained it end-to-end on real text.
- **Masked mean pooling** is pad-safe, length-robust, and gives every position a direct gradient path; it is the right default before attention.
- Same recipe, three cores: **GRU ≈ LSTM** in accuracy, with the GRU's recurrent core ~25% leaner; the **vanilla RNN trails**, and the flip review shows one reason why.
- Everything here transfers to transformers: keep the skeleton, swap the mixer for attention.